# Multi-Agent Systems: A Supervisor

This notebook builds a small multi-agent system with plain LangGraph: one supervisor
graph node that decides which specialist should act next, a researcher agent (built
with `create_agent` and a small lookup tool) that gathers information, and a writer
node that turns that information into a final answer. Each specialist hands control
back to the supervisor with `Command(goto="supervisor", ...)` instead of the
supervisor having to guess what happened. A step counter stops the graph even if the
routing decision never picks "finish".

In [ ]:
import operator
from typing import Annotated, Literal, TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command

In [ ]:
load_dotenv()

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

## Why Split Work Across Agents

A single agent with one long system prompt and a big tool list works for small
tasks, but it gets harder to control as it grows: the prompt has to cover every
role at once, and any tool is available at every step even when it does not apply.
Splitting the work into specialist agents keeps each agent's prompt and tool list
small and focused, and makes the overall flow easier to reason about, test, and
reuse.

There are two common patterns for connecting specialist agents:

- **Supervisor**: a coordinator node looks at the task, decides which specialist
  should act next, and always gets control back after that specialist finishes.
  This is what this notebook builds.
- **Handoffs**: agents transfer control directly to each other, without a central
  coordinator. The `Multi_Agent_Guide.md` file next to this notebook covers this
  pattern in more detail.

A compiled agent (from `create_agent`, or a hand-built `StateGraph`) can be called
from inside another graph's node just like any other function. That is the same
idea as `19_Subgraphs`: a whole graph used as one step of a bigger graph.

## Part 1: A Tool for the Researcher

The researcher agent gets one small tool: a lookup over a few hardcoded facts about
a fictional product. This keeps the example self-contained, with no network calls.

In [ ]:
PRODUCT_FACTS = {
    "battery": "The Aurora Smart Thermostat runs for about 8 months per battery charge.",
    "price": "The Aurora Smart Thermostat costs $89.",
    "compatibility": "The Aurora Smart Thermostat works with most 24V HVAC systems and needs a C-wire.",
}


@tool
def lookup_fact(topic: str) -> str:
    """Look up a fact about the Aurora Smart Thermostat. Topic is one of: battery, price, compatibility."""
    return PRODUCT_FACTS.get(topic.lower(), "No fact found for that topic.")

## Part 2: The Researcher Agent

The researcher is a normal prebuilt agent from `16_Prebuilt_Agents`, scoped to one
job with a short system prompt and one tool.

In [ ]:
researcher_agent = create_agent(
    model=llm,
    tools=[lookup_fact],
    system_prompt="You are a researcher. Use lookup_fact to answer the request, then summarize what you found in one or two sentences.",
)

## Part 3: Shared Graph State

All three nodes (supervisor, researcher, writer) read and write the same state.
`history` uses `operator.add` as a reducer so each node's contribution is appended
to the list instead of overwriting it, the same reducer pattern used for messages
elsewhere in this repo.

In [ ]:
class SupervisorState(TypedDict, total=False):
    task: str
    research_notes: str
    final_answer: str
    steps: int
    history: Annotated[list[str], operator.add]

## Part 4: The Supervisor Node

The supervisor asks the model to choose the next worker with structured output, then
returns a `Command` that both records the decision and routes to it, the same
`Command(goto=...)` pattern from `06_Command_Routing`. A hard step limit
(`MAX_STEPS`) guarantees the graph ends even if the model keeps choosing a worker
instead of `FINISH`.

In [ ]:
class RouteDecision(BaseModel):
    next: Literal["researcher", "writer", "FINISH"]


MAX_STEPS = 6

In [ ]:
def supervisor(state: SupervisorState) -> Command[Literal["researcher", "writer", "__end__"]]:
    if state.get("steps", 0) >= MAX_STEPS:
        return Command(
            goto=END,
            update={"history": ["supervisor: step limit reached, stopping"]},
        )

    if state.get("final_answer"):
        return Command(goto=END, update={"history": ["supervisor: final answer ready, finishing"]})

    prompt = (
        f"Task: {state['task']}\n"
        f"Research notes so far: {state.get('research_notes') or 'none yet'}\n"
        f"Final answer so far: {state.get('final_answer') or 'none yet'}\n"
        "Choose 'researcher' if there are no research notes yet, 'writer' if there are "
        "research notes but no final answer yet, or 'FINISH' if a final answer already exists."
    )
    decision = llm.with_structured_output(RouteDecision).invoke(prompt)

    if decision.next == "FINISH":
        return Command(goto=END, update={"history": ["supervisor: chose FINISH"]})

    return Command(
        goto=decision.next,
        update={"steps": state.get("steps", 0) + 1, "history": [f"supervisor: routed to {decision.next}"]},
    )

## Part 5: The Worker Nodes

Each worker does its job and then routes straight back to `"supervisor"`. Neither
worker decides what happens next; that decision always goes through the supervisor.

In [ ]:
def researcher_node(state: SupervisorState) -> Command[Literal["supervisor"]]:
    result = researcher_agent.invoke(
        {"messages": [{"role": "user", "content": f"Research this for the task: {state['task']}"}]}
    )
    notes = result["messages"][-1].content
    return Command(
        goto="supervisor",
        update={"research_notes": notes, "history": ["researcher: gathered notes"]},
    )


def writer_node(state: SupervisorState) -> Command[Literal["supervisor"]]:
    prompt = (
        f"Task: {state['task']}\n"
        f"Research notes: {state.get('research_notes')}\n"
        "Write a short final answer using only these notes."
    )
    response = llm.invoke(prompt)
    return Command(
        goto="supervisor",
        update={"final_answer": response.content, "history": ["writer: wrote final answer"]},
    )

## Part 6: Building the Graph

Every routing decision above already came back as a `Command(goto=...)`, so the
graph only needs its nodes and one starting edge. There is no
`add_conditional_edges` call anywhere in this notebook.

In [ ]:
graph = StateGraph(SupervisorState)
graph.add_node("supervisor", supervisor)
graph.add_node("researcher", researcher_node)
graph.add_node("writer", writer_node)
graph.add_edge(START, "supervisor")

app = graph.compile()

## Part 7: Running It

The task asks about something the researcher's tool actually knows, so a full run
should visit `researcher` at least once, then `writer`, then finish.

In [ ]:
result = app.invoke({
    "task": "How long does the Aurora Smart Thermostat's battery last?",
    "steps": 0,
})
print("Route taken:")
for line in result["history"]:
    print(" -", line)
print()
print("Final answer:", result.get("final_answer"))

## Part 8: Why the Step Limit Matters

Structured output is a model prediction, not a guarantee. A model can keep choosing
`researcher` instead of ever moving on, especially with a very deterministic or
low-effort model. Without `MAX_STEPS`, that would be an infinite loop between
`supervisor` and `researcher`. With it, the graph always ends, even in the worst
case, and `history` shows exactly what happened, including a run that never reached
`writer`.

## Summary

A supervisor graph keeps one node in charge of routing (`supervisor`), while
specialist nodes (`researcher`, `writer`) each do one job and hand control back with
`Command(goto="supervisor", ...)`. `Command` removes the need for
`add_conditional_edges` here: every routing decision is made and returned from
inside the node that made it, the same idea as `06_Command_Routing`, just with more
than two possible destinations. A specialist agent built with `create_agent` slots
into a node exactly like a plain Python function does, the same way a compiled graph
slots into a parent graph in `19_Subgraphs`. And because the routing decision comes
from a model, a hard step limit is what actually guarantees the graph terminates.